# 02 — Silver: typed, cleaned, deduplicated

Where judgement gets applied. Four things happen:

1. Strings become real types; the literal `'null'` becomes a true NULL
2. Sentinel dates are resolved
3. Duplicate client rows (contract renewals) are collapsed
4. **`is_bulk_load` is derived** — the single transformation that changes
   every conclusion downstream

Redundant string date columns (`created_at_str`, `closed_at_str`) are dropped
here rather than at ingestion, so Bronze stays a faithful copy of the source.

In [0]:
from pyspark.sql import functions as F, Window

spark.sql("CREATE SCHEMA IF NOT EXISTS silver.bid")

bronze_bids = spark.table("bronze.bid.bids")
bronze_clients = spark.table("bronze.bid.clients")

## Null handling

The export writes the four-character string `'null'`, and `'-'` for an
absent closure date. Neither is a NULL to Spark, so both would silently
survive every downstream filter.

In [0]:
def denull(df, placeholders=("null", "-", "")):
    """Replace placeholder strings with true NULLs across all string columns."""
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(
                c, F.when(F.trim(F.col(c)).isin(list(placeholders)), None)
                    .otherwise(F.col(c))
            )
    return df


bids = denull(bronze_bids)
clients = denull(bronze_clients)

## Typing

`outcome` is deliberately left nullable: NULL means the bid is still open,
which is a distinct state from lost and must not collapse into `0`. Roughly
28% of the table sits in that state.

`to_timestamp` is given an explicit format rather than left to infer. The
default parser is version-dependent and returns NULL on a miss instead of
raising — exactly the kind of silent data loss this pipeline is built to
avoid. The assert right after fails loudly if that ever happens.

In [0]:
TS_FORMAT = "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"

bids_typed = (
    bids
    .withColumn("bid_id", F.col("bid_id").cast("long"))
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("created_at", F.to_timestamp("created_at", TS_FORMAT))
    .withColumn("bid_date", F.to_timestamp("bid_date", TS_FORMAT))
    .withColumn("closed_at", F.to_timestamp("closed_at", TS_FORMAT))
    .withColumn("outcome", F.col("outcome").cast("int"))          # NULL = open
    .withColumn("is_confirmed_date", F.col("is_confirmed_date").cast("int"))
    .withColumn("contract_value_brl", F.col("contract_value_brl").cast("double"))
    .drop("created_at_str", "closed_at_str")
)

# Guard: a created_at that was non-NULL before the cast but NULL after it
# means the format string stopped matching the source — fail loudly, not
# silently.
bad_created = (
    bids.filter(F.col("created_at").isNotNull())
    .join(bids_typed.filter(F.col("created_at").isNull()), "bid_id", "inner")
    .count()
)
assert bad_created == 0, (
    f"{bad_created} created_at values failed to parse with format {TS_FORMAT} "
    "— check for a source date format change."
)

## Deriving `is_bulk_load`

A large share of rows were mass-imported during a system migration rather
than entered as bids happened. They share an identical `created_at` down to
the second, and they behave nothing like organic bids — a far lower win rate,
concentrated in specific portfolios.

Left unflagged, they poison every segmented metric: the executive who owned
the migrated portfolio looks like the worst performer in the company purely
because of how their records were loaded.

The threshold of 10 is a judgement call. Two bids registered in the same
second is plausible; ten is not.

In [0]:
BULK_THRESHOLD = 10

batch = Window.partitionBy("created_at")

bids_flagged = (
    bids_typed
    .withColumn("_batch_size", F.count("*").over(batch))
    .withColumn("is_bulk_load", (F.col("_batch_size") >= BULK_THRESHOLD).cast("boolean"))
    .drop("_batch_size")
)

## Loss reason coverage

`competitor_name` carries a default value written whenever nobody completed
the post-mortem. Flagging it explicitly stops it being counted as a real
competitor in any downstream aggregate — which would otherwise produce the
false headline that one competitor takes the overwhelming majority of losses.

In [0]:
PLACEHOLDER_COMPETITOR = "Competitor 1"

bids_clean = (
    bids_flagged
    .withColumn(
        "competitor_is_placeholder",
        (F.col("competitor_name") == F.lit(PLACEHOLDER_COMPETITOR)).cast("boolean"),
    )
    .withColumn("has_loss_reason", F.col("loss_reason").isNotNull())
    .withColumn(
        "bid_status",
        F.when(F.col("outcome") == 1, "won")
         .when(F.col("outcome") == 0, "lost")
         .otherwise("open"),
    )
)

bids_clean.write.format("delta").mode("overwrite").saveAsTable("silver.bid.bids_clean")

## Clients: sentinel dates and duplicates

`2999-12-31` marks an open-ended contract; keeping it as a date would put a
977-year contract into any duration calculation. It becomes NULL alongside an
explicit `is_open_ended` flag.

Renewals are recorded as a second row for the same `client_id`. Deduplicating
on the most recent contract keeps one row per client, which is what the join
to `bids` requires — an un-deduplicated dimension would fan out the fact table
and inflate every count.

In [0]:
SENTINEL = "2999-12-31"

clients_typed = (
    clients
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("is_open_ended", (F.col("end_date").startswith(SENTINEL)).cast("boolean"))
    .withColumn(
        "end_date",
        F.when(F.col("end_date").startswith(SENTINEL), None)
         .otherwise(F.to_date("end_date")),
    )
    .withColumn("start_date", F.to_date("start_date"))
)

latest = Window.partitionBy("client_id").orderBy(F.col("start_date").desc_nulls_last())

clients_clean = (
    clients_typed
    .withColumn("_rn", F.row_number().over(latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

clients_clean.write.format("delta").mode("overwrite").saveAsTable("silver.bid.clients_clean")

## Referential integrity

A bid pointing at a client that does not exist would be dropped silently by
an inner join later. Checking here means it surfaces as a number, not as a
quietly shrinking row count.

In [0]:
orphans = (
    bids_clean.join(clients_clean, "client_id", "left_anti").count()
)
dupes = (
    clients_clean.groupBy("client_id").count().filter("count > 1").count()
)

print(f"orphan bids: {orphans}")
print(f"duplicate client_ids after dedup: {dupes}")

assert dupes == 0, "clients_clean must have one row per client_id"

In [0]:
%sql
select * from silver.bid.bids_clean

In [0]:
%sql
select * from silver.bid.clients_clean